In [1]:
import pandas as pd

df = pd.read_excel("D:/sales_data_analysis/data/raw/sales-transactions-100.xlsx")

In [3]:
df.head(5)

,date,region,product,channel,units,unit_price,revenue
0,2024-07-01,LATAM,Desk Lamp,Retail,36,64.22,2311.92
1,2024-07-08,EMEA,HD Webcam,Partner,28,26.17,732.76
2,2024-07-16,North America,Desk Lamp,Retail,19,64.22,1220.18
3,2024-07-23,EMEA,Mechanical Keyboard,Online,38,210.43,7996.34
4,2024-07-30,LATAM,Portable SSD,Online,3,41.29,123.87


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        100 non-null    str    
 1   region      100 non-null    str    
 2   product     100 non-null    str    
 3   channel     100 non-null    str    
 4   units       100 non-null    int64  
 5   unit_price  100 non-null    float64
 6   revenue     100 non-null    float64
dtypes: float64(2), int64(1), str(4)
memory usage: 5.6 KB


In [6]:
df.shape

(100, 7)

In [8]:
dup = df.duplicated().sum()
print(f"Number of duplicate rows: {dup}")

Number of duplicate rows: 0


Revenue Validation

In [38]:
# Calculate expected revenue using Units × Unit_Price.
df['Expected_Revenue'] = df['units'] * df['unit_price']
print(df[['units', 'unit_price', 'Expected_Revenue']].head(5))

# Compare expected revenue with the existing Revenue column.
df['Revenue_mismatch'] = df['Expected_Revenue'].round(2) != df['revenue'].round(2)

# Identify how many records have mismatched revenue.
mismatch_count = df['Revenue_mismatch'].sum()
print(f"Number of mismatched revenue records: {mismatch_count}")


   units  unit_price  Expected_Revenue
0     36       64.22           2311.92
1     28       26.17            732.76
2     19       64.22           1220.18
3     38      210.43           7996.34
4      3       41.29            123.87
Number of mismatched revenue records: 0


Unusual values

In [40]:
# Investigation	Condition
# Zero units	units == 0
zero_unit = df[df['units'] == 0]
print("zero Units: ", len(zero_unit))
# Negative units	units < 0
neg_unit = df[df['units'] < 0]
print("Negative Units: ", len(neg_unit))
# Zero unit price	unit_price == 0
zero_price = df[df['unit_price'] == 0]
print("Zero Unit Price: ", len(zero_price))
# Negative unit price	unit_price < 0
neg_price = df[df['unit_price'] < 0]
print("Negative Unit Price: ", len(neg_price))
# Negative revenue	revenue < 0
neg_revenue = df[df['revenue'] < 0]
print("Negative Revenue: ", len(neg_revenue))

zero Units:  0
Negative Units:  0
Zero Unit Price:  0
Negative Unit Price:  0
Negative Revenue:  0


Business analysis

1. Which product generates the highest total revenue for our company?

In [4]:
df.info()
# First, group by which column?
same_product = df.groupby('product')['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
print(same_product)

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        100 non-null    str    
 1   region      100 non-null    str    
 2   product     100 non-null    str    
 3   channel     100 non-null    str    
 4   units       100 non-null    int64  
 5   unit_price  100 non-null    float64
 6   revenue     100 non-null    float64
dtypes: float64(2), int64(1), str(4)
memory usage: 5.6 KB
               product   revenue
3  Mechanical Keyboard  32616.65
6            USB-C Hub  32431.84
0            Desk Lamp  16247.66
4         Portable SSD  15937.94
1            HD Webcam   8505.25
2         Laptop Stand   5985.90
5        Standing Desk   4659.14
7       Wireless Mouse   2847.04


Which region contributes the highest total revenue?

In [24]:
high_region_revenue = df.groupby('region')['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
print(high_region_revenue)

          region   revenue
4  North America  32212.07
0            ANZ  29255.40
2           EMEA  22152.22
1           APAC  19718.50
3          LATAM  15893.23


Which sales channel generates the highest total revenue?

In [3]:
high_channel_revenue = df.groupby('channel')['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
print(high_channel_revenue)

   channel   revenue
0   Online  45318.89
2   Retail  39448.39
1  Partner  34464.14


Monthly Sales Analysis

Which month has the highest total revenue?

In [6]:
# convert the date column to datetime format
df['date'] = pd.to_datetime(df['date'], errors='coerce')
# df.info()

extract_month = df['date'].dt.month
highest_month_revenue  = df.groupby(extract_month)['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False)
print(highest_month_revenue)

    date   revenue
8      9  24181.03
6      7  18312.30
1      2  12017.03
10    11  10495.77
5      6   8827.36
0      1   8686.46
2      3   8645.86
7      8   8520.82
11    12   5940.40
9     10   5819.01
4      5   4314.28
3      4   3471.10


In [13]:
products_highest_saled = df.groupby('product')['units'].sum().reset_index().sort_values(by='units', ascending=False)
print(products_highest_saled)

               product  units
2         Laptop Stand    405
4         Portable SSD    386
1            HD Webcam    325
6            USB-C Hub    322
0            Desk Lamp    253
3  Mechanical Keyboard    155
7       Wireless Mouse     82
5        Standing Desk     29


What is the average selling price of each product?

In [15]:
Avg_selling_price = df.groupby('product')['unit_price'].mean().reset_index().sort_values(by='unit_price', ascending=False)
print(Avg_selling_price)

               product  unit_price
3  Mechanical Keyboard      210.43
5        Standing Desk      160.66
6            USB-C Hub      100.72
0            Desk Lamp       64.22
4         Portable SSD       41.29
7       Wireless Mouse       34.72
1            HD Webcam       26.17
2         Laptop Stand       14.78


What percentage of total revenue comes from each region?

In [19]:
percent_revenue_per_region = df.groupby('region')['revenue'].sum().reset_index()
percent_revenue_per_region['percent_revenue'] = (percent_revenue_per_region['revenue'] / percent_revenue_per_region['revenue'].sum()) * 100
print(percent_revenue_per_region.sort_values(by='percent_revenue', ascending=False))

          region   revenue  percent_revenue
4  North America  32212.07        27.016427
0            ANZ  29255.40        24.536653
2           EMEA  22152.22        18.579180
1           APAC  19718.50        16.538007
3          LATAM  15893.23        13.329733


Which region and product combination generates the highest revenue?

In [22]:
region_product_revenue = df.groupby(['region', 'product'])['revenue'].sum().reset_index().sort_values(by='revenue', ascending=False).head(3)
print(region_product_revenue)


           region              product   revenue
28  North America  Mechanical Keyboard  11784.08
15           EMEA  Mechanical Keyboard  11573.65
6             ANZ            USB-C Hub  11079.20


– Find the Top 5 Highest Sales Transactions

In [27]:
top_5_products = df.nlargest(5, 'revenue')[['product', 'revenue', 'date', 'region', 'units']]
print(top_5_products)

                product  revenue       date         region  units
10  Mechanical Keyboard  8417.20 2024-09-13            ANZ     40
3   Mechanical Keyboard  7996.34 2024-07-23           EMEA     38
6   Mechanical Keyboard  6523.33 2024-08-14  North America     31
12  Mechanical Keyboard  5260.75 2024-09-27  North America     25
86            USB-C Hub  4028.80 2026-03-26           APAC     40


What are the overall sales KPIs?

In [34]:
# Total Revenue
total_revenue = df['revenue'].sum()
print(f"Total Revenue: {total_revenue}")

# Total Units Sold
total_units_sold = df['units'].sum()
print(f"Total Units Sold: {total_units_sold}")

# Total Number of Transactions
total_transactions = len(df)
print(f"Total Number of Transactions: {total_transactions}")

# Average Transaction Revenue
average_transaction_revenue = df['revenue'].mean()
print(f"Average Transaction Revenue: {average_transaction_revenue}")

# Number of Unique Products
unique_products = df['product'].nunique()
print(f"Number of Unique Products: {unique_products}")

Total Revenue: 119231.42
Total Units Sold: 1957
Total Number of Transactions: 100
Average Transaction Revenue: 1192.3142
Number of Unique Products: 8
